# Deep Learning-Based Prediction of Compression After Impact (CAI) Strength in CFRP Laminates

**Research-Grade Implementation — IEEE Journal Level**

---
**Abstract:**  
This notebook presents a transfer-learning pipeline using MobileNetV2 to predict the Compression After Impact (CAI) strength of Carbon Fibre Reinforced Polymer (CFRP) specimens from surface images. The workflow includes rigorous data preprocessing, stratified splitting, label-leak-free normalization, learning-rate scheduling, early stopping, and comprehensive evaluation metrics (MAE, RMSE, R², MAPE) suitable for IEEE publication.

---

## 1. Environment Setup & Reproducibility

In [ ]:
# ── Reproducibility ────────────────────────────────────────────────────────────
import os, random, warnings
import numpy as np
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Suppress minor warnings for clean output
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

print(f"TensorFlow version : {tf.__version__}")
print(f"GPU available      : {len(tf.config.list_physical_devices('GPU')) > 0}")

## 2. Mount Google Drive & Extract Dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile

ZIP_PATH     = "/content/drive/MyDrive/CRPF_IMAGES.zip"
EXTRACT_PATH = "/content/dataset"

if not os.path.exists(EXTRACT_PATH):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(EXTRACT_PATH)
    print("Dataset extracted.")
else:
    print("Dataset already extracted — skipping.")

# Quick directory scan
for root, dirs, files in os.walk(EXTRACT_PATH):
    print(f"{root}  →  {len(files)} file(s)")

## 3. Load & Clean Metadata (Excel Labels)

In [ ]:
import pandas as pd

EXCEL_PATH = (
    "/content/dataset/5_Compression after impact strength/"
    "Compression after impact strength_v0.2.xlsx"
)

# ── Read raw sheet ─────────────────────────────────────────────────────────────
raw = pd.read_excel(EXCEL_PATH, header=None)
print("Raw shape:", raw.shape)
print(raw.head(6))

In [ ]:
# ── Robust header detection ─────────────────────────────────────────────────────
# Skip rows until we find the numeric data block; drop unit rows automatically.
df = pd.read_excel(EXCEL_PATH, header=None, skiprows=2)  # adjust if needed

df.columns = ["_drop", "Specimen_ID", "CAI_strength_MPa", "Reduction_pct"]
df = df.drop(columns=["_drop"])
df = df.dropna(subset=["Specimen_ID", "CAI_strength_MPa"])   # remove blank rows
df = df.reset_index(drop=True)

# ── Normalise Specimen_ID: lowercase, strip trailing 't', spaces ───────────────
df["Specimen_ID"] = (
    df["Specimen_ID"]
    .astype(str)
    .str.strip()
    .str.lower()
    .str.rstrip('t')
)

# ── Coerce strength to numeric ─────────────────────────────────────────────────
df["CAI_strength_MPa"] = pd.to_numeric(df["CAI_strength_MPa"], errors='coerce')
df = df.dropna(subset=["CAI_strength_MPa"]).reset_index(drop=True)

print(f"Total valid specimens in Excel: {len(df)}")
print(df.describe())

## 4. Match Images to Labels

In [ ]:
IMG_FOLDER = "/content/dataset/3_Specimen image"

VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def clean_image_key(filename: str) -> str:
    """Strip extension + suffixes (_front / _back) and lowercase."""
    name = os.path.splitext(filename)[0]   # remove extension
    name = name.replace("_front", "").replace("_back", "")
    return name.strip().lower()

# Build lookup dict: cleaned_key → full image path
img_lookup: dict[str, list[str]] = {}
for fname in os.listdir(IMG_FOLDER):
    ext = os.path.splitext(fname)[1].lower()
    if ext not in VALID_EXTS:
        continue
    key = clean_image_key(fname)
    img_lookup.setdefault(key, []).append(os.path.join(IMG_FOLDER, fname))

# Match: one record per image (front + back treated as separate samples)
records = []  # list of {img_path, specimen_id, strength_MPa}
unmatched = []

for _, row in df.iterrows():
    sid = row["Specimen_ID"]
    if sid in img_lookup:
        for path in img_lookup[sid]:
            records.append({
                "img_path"      : path,
                "specimen_id"   : sid,
                "strength_MPa"  : float(row["CAI_strength_MPa"])
            })
    else:
        unmatched.append(sid)

dataset_df = pd.DataFrame(records)

print(f"Matched image-label pairs : {len(dataset_df)}")
print(f"Unmatched specimen IDs    : {len(unmatched)}  {unmatched[:5]}")
print(dataset_df.head())

## 5. Exploratory Data Analysis (EDA)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2

sns.set_theme(style="whitegrid", font_scale=1.2)

strengths = dataset_df["strength_MPa"].values

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Histogram + KDE
sns.histplot(strengths, bins=20, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Distribution of CAI Strength", fontweight='bold')
axes[0].set_xlabel("CAI Strength (MPa)")
axes[0].set_ylabel("Count")

# 2. Box plot
axes[1].boxplot(strengths, vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[1].set_title("Box Plot of CAI Strength", fontweight='bold')
axes[1].set_ylabel("CAI Strength (MPa)")
axes[1].set_xticks([])

# 3. Cumulative distribution
sorted_s = np.sort(strengths)
cdf = np.arange(1, len(sorted_s)+1) / len(sorted_s)
axes[2].plot(sorted_s, cdf, lw=2, color='darkorange')
axes[2].set_title("Empirical CDF", fontweight='bold')
axes[2].set_xlabel("CAI Strength (MPa)")
axes[2].set_ylabel("Cumulative Probability")

plt.suptitle("CFRP CAI Strength — Exploratory Data Analysis",
             fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig("eda.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\nStrength stats (MPa):")
print(f"  Min   = {strengths.min():.2f}")
print(f"  Max   = {strengths.max():.2f}")
print(f"  Mean  = {strengths.mean():.2f}")
print(f"  Std   = {strengths.std():.2f}")
print(f"  Count = {len(strengths)}")

In [ ]:
# ── Sample image grid ─────────────────────────────────────────────────────────
sample_rows = dataset_df.sample(6, random_state=SEED)

fig, axes = plt.subplots(2, 3, figsize=(14, 9))

for ax, (_, row) in zip(axes.flat, sample_rows.iterrows()):
    img = cv2.cvtColor(cv2.imread(row["img_path"]), cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(f"{row['specimen_id']}\n{row['strength_MPa']:.1f} MPa",
                 fontsize=10)
    ax.axis('off')

plt.suptitle("Sample CFRP Specimens with CAI Strength Labels",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("sample_images.png", dpi=150, bbox_inches='tight')
plt.show()

## 6. Stratified Train / Validation / Test Split

> **Critical fix:** Label normalisation (MinMaxScaler) is fitted **only on the training set** to prevent data leakage into validation and test sets.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

# ── Stratified split via strength quantile bins ────────────────────────────────
dataset_df["_strat_bin"] = pd.qcut(
    dataset_df["strength_MPa"], q=5, labels=False, duplicates='drop'
)

train_df, temp_df = train_test_split(
    dataset_df, test_size=0.20, random_state=SEED,
    stratify=dataset_df["_strat_bin"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=SEED,
    stratify=temp_df["_strat_bin"]
)

for name, subset in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"{name:5s}: {len(subset):4d} samples | "
          f"μ={subset['strength_MPa'].mean():.2f} MPa | "
          f"σ={subset['strength_MPa'].std():.2f} MPa")

# ── Fit scaler ONLY on training labels (no leakage) ───────────────────────────
scaler = MinMaxScaler()
y_train_raw = train_df["strength_MPa"].values.reshape(-1, 1)
y_val_raw   = val_df["strength_MPa"].values.reshape(-1, 1)
y_test_raw  = test_df["strength_MPa"].values.reshape(-1, 1)

y_train = scaler.fit_transform(y_train_raw)   # fit here only
y_val   = scaler.transform(y_val_raw)
y_test  = scaler.transform(y_test_raw)

## 7. Image Loading & Augmentation Pipeline

In [ ]:
IMG_SIZE  = 224
BATCH_SIZE = 16

# ── Image loading ──────────────────────────────────────────────────────────────
def load_images(paths: list) -> np.ndarray:
    """Load, resize, and normalise to [0, 1]."""
    imgs = []
    for p in paths:
        img = cv2.imread(p)
        if img is None:
            raise FileNotFoundError(f"Cannot read: {p}")
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMG_SIZE, IMG_SIZE),
                         interpolation=cv2.INTER_AREA)
        imgs.append(img.astype(np.float32) / 255.0)
    return np.array(imgs, dtype=np.float32)

print("Loading training images   …", end=" ")
X_train = load_images(train_df["img_path"].tolist()); print(X_train.shape)

print("Loading validation images …", end=" ")
X_val = load_images(val_df["img_path"].tolist()); print(X_val.shape)

print("Loading test images       …", end=" ")
X_test = load_images(test_df["img_path"].tolist()); print(X_test.shape)

In [ ]:
# ── tf.data pipeline with augmentation (train only) ───────────────────────────
AUTOTUNE = tf.data.AUTOTUNE

@tf.function
def augment(image, label):
    """Light augmentation relevant for microscopy / specimen images."""
    image = tf.image.random_flip_left_right(image)
    image = tf.image.random_flip_up_down(image)
    image = tf.image.random_brightness(image, max_delta=0.10)
    image = tf.image.random_contrast(image, lower=0.85, upper=1.15)
    # Random 0°/90°/180°/270° rotation
    k = tf.random.uniform(shape=[], minval=0, maxval=4, dtype=tf.int32)
    image = tf.image.rot90(image, k=k)
    image = tf.clip_by_value(image, 0.0, 1.0)
    return image, label

def make_dataset(X, y, augment_data=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((X, y))
    if shuffle:
        ds = ds.shuffle(buffer_size=len(X), seed=SEED)
    if augment_data:
        ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(X_train, y_train, augment_data=True,  shuffle=True)
val_ds   = make_dataset(X_val,   y_val,   augment_data=False, shuffle=False)
test_ds  = make_dataset(X_test,  y_test,  augment_data=False, shuffle=False)

print("Datasets ready.")

## 8. Model Architecture — MobileNetV2 with Custom Regression Head

In [ ]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, CSVLogger
)

def build_model(freeze_base: bool = True) -> tf.keras.Model:
    base = MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base.trainable = not freeze_base

    inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    # MobileNetV2 expects inputs in [-1, 1]; apply preprocessing
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
    x = base(x, training=False)  # training=False keeps BN in inference mode
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dense(256, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(128, activation='relu',
                     kernel_regularizer=regularizers.l2(1e-4))(x)
    x = layers.Dropout(0.3)(x)
    output = layers.Dense(1, activation='linear', name='cai_strength')(x)

    model = models.Model(inputs, output)
    return model

model = build_model(freeze_base=True)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='huber',          # robust to outliers vs pure MSE
    metrics=['mae', tf.keras.metrics.RootMeanSquaredError(name='rmse')]
)

model.summary()

## 9. Phase 1 — Feature Extraction (Frozen Base)

In [ ]:
callbacks_phase1 = [
    EarlyStopping(monitor='val_loss', patience=8,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=4, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_phase1.weights.h5',
                    monitor='val_loss', save_best_only=True,
                    save_weights_only=True, verbose=0),
    CSVLogger('training_phase1.csv')
]

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks_phase1,
    verbose=1
)

print("Phase 1 complete.")

## 10. Phase 2 — Fine-Tuning (Unfreeze Top Layers)

In [ ]:
# Unfreeze only the last 40 layers of the base
base_model = model.layers[3]   # MobileNetV2 block
base_model.trainable = True

UNFREEZE_FROM = len(base_model.layers) - 40
for layer in base_model.layers[:UNFREEZE_FROM]:
    layer.trainable = False

trainable_count = sum(1 for l in model.layers if l.trainable)
print(f"Trainable layers after fine-tune setup: {trainable_count}")

# Recompile with much lower LR
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='huber',
    metrics=['mae', tf.keras.metrics.RootMeanSquaredError(name='rmse')]
)

callbacks_phase2 = [
    EarlyStopping(monitor='val_loss', patience=10,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                      patience=5, min_lr=1e-8, verbose=1),
    ModelCheckpoint('best_phase2.weights.h5',
                    monitor='val_loss', save_best_only=True,
                    save_weights_only=True, verbose=0),
    CSVLogger('training_phase2.csv')
]

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=40,
    callbacks=callbacks_phase2,
    verbose=1
)

print("Phase 2 complete.")

## 11. Training History Visualisation

In [ ]:
def merge_histories(h1, h2):
    """Concatenate two Keras History objects into one dict."""
    combined = {}
    for k in h1.history:
        combined[k] = h1.history[k] + h2.history[k]
    return combined

hist = merge_histories(history1, history2)
phase1_end = len(history1.history['loss'])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

metric_map = [
    ('loss',  'Huber Loss',  'Loss'),
    ('mae',   'MAE (scaled)','MAE'),
    ('rmse',  'RMSE (scaled)','RMSE'),
]

for ax, (key, title, ylabel) in zip(axes, metric_map):
    epochs = range(1, len(hist[key]) + 1)
    ax.plot(epochs, hist[key],     label='Train', lw=2)
    ax.plot(epochs, hist[f'val_{key}'], label='Val',   lw=2, ls='--')
    ax.axvline(phase1_end, color='gray', ls=':', alpha=0.7, label='Fine-tune start')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.legend()

plt.suptitle("Training History — Phase 1 + Phase 2",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("training_curves.png", dpi=150, bbox_inches='tight')
plt.show()

## 12. Evaluation on Test Set — Full Metrics Suite

In [ ]:
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score
)

# ── Predict and inverse-transform ─────────────────────────────────────────────
y_pred_scaled = model.predict(test_ds, verbose=0)           # shape (N, 1)

# Clip to valid scaler range before inverse transform
y_pred_scaled = np.clip(y_pred_scaled, 0.0, 1.0)

y_pred_mpa  = scaler.inverse_transform(y_pred_scaled).flatten()
y_true_mpa  = scaler.inverse_transform(y_test).flatten()

# ── Compute metrics ───────────────────────────────────────────────────────────
mae   = mean_absolute_error(y_true_mpa, y_pred_mpa)
mse   = mean_squared_error(y_true_mpa,  y_pred_mpa)
rmse  = np.sqrt(mse)
r2    = r2_score(y_true_mpa, y_pred_mpa)

# Mean Absolute Percentage Error (guarded against zero-divide)
eps  = 1e-8
mape = np.mean(np.abs((y_true_mpa - y_pred_mpa) / (y_true_mpa + eps))) * 100

# Normalised RMSE
nrmse = rmse / (y_true_mpa.max() - y_true_mpa.min()) * 100

print("="*50)
print("  TEST SET PERFORMANCE METRICS")
print("="*50)
print(f"  MAE   = {mae:.4f} MPa")
print(f"  RMSE  = {rmse:.4f} MPa")
print(f"  NRMSE = {nrmse:.2f} %")
print(f"  MAPE  = {mape:.2f} %")
print(f"  R²    = {r2:.4f}")
print("="*50)

## 13. Publication-Quality Evaluation Plots

In [ ]:
from scipy import stats

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ── 1. Parity Plot (Predicted vs Actual) ──────────────────────────────────────
ax = axes[0]
lo = min(y_true_mpa.min(), y_pred_mpa.min())
hi = max(y_true_mpa.max(), y_pred_mpa.max())

ax.scatter(y_true_mpa, y_pred_mpa, alpha=0.7, edgecolors='k',
           linewidths=0.4, c='steelblue', s=60, label='Test samples')
ax.plot([lo, hi], [lo, hi], 'r--', lw=2, label='Perfect fit')

# ±10 % bands
ax.plot([lo, hi], [lo*1.10, hi*1.10], 'g:', lw=1.5, label='±10 %')
ax.plot([lo, hi], [lo*0.90, hi*0.90], 'g:', lw=1.5)

ax.set_xlabel("Actual CAI Strength (MPa)", fontsize=12)
ax.set_ylabel("Predicted CAI Strength (MPa)", fontsize=12)
ax.set_title(f"Parity Plot  (R² = {r2:.4f})", fontweight='bold')
ax.legend(fontsize=9)

# Add regression line
slope, intercept, *_ = stats.linregress(y_true_mpa, y_pred_mpa)
x_line = np.linspace(lo, hi, 100)
ax.plot(x_line, slope*x_line + intercept, 'b-', lw=1.5,
        label=f'OLS  y={slope:.2f}x+{intercept:.1f}')
ax.legend(fontsize=9)

# ── 2. Residuals Plot ─────────────────────────────────────────────────────────
residuals = y_true_mpa - y_pred_mpa
ax = axes[1]
ax.scatter(y_pred_mpa, residuals, alpha=0.7, edgecolors='k',
           linewidths=0.4, c='darkorange', s=60)
ax.axhline(0,  color='red', lw=2, ls='--')
ax.axhline(+2*residuals.std(), color='gray', lw=1, ls=':')
ax.axhline(-2*residuals.std(), color='gray', lw=1, ls=':',
           label='±2σ band')
ax.set_xlabel("Predicted CAI Strength (MPa)", fontsize=12)
ax.set_ylabel("Residual (MPa)", fontsize=12)
ax.set_title("Residual Plot", fontweight='bold')
ax.legend(fontsize=9)

# ── 3. Error Distribution ─────────────────────────────────────────────────────
ax = axes[2]
sns.histplot(residuals, bins=20, kde=True, ax=ax, color='mediumpurple')
ax.axvline(0, color='red', lw=2, ls='--', label='Zero error')
ax.set_xlabel("Prediction Error (MPa)", fontsize=12)
ax.set_ylabel("Frequency", fontsize=12)
ax.set_title("Error Distribution", fontweight='bold')
ax.legend(fontsize=9)

plt.suptitle("CFRP CAI Strength Prediction — Evaluation Plots",
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig("evaluation_plots.png", dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Q-Q plot for normality of residuals ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
(osm, osr), (slope, intercept, r) = stats.probplot(residuals, dist='norm')
ax.scatter(osm, osr, s=30, alpha=0.7, edgecolors='k', lw=0.3, c='steelblue')
ax.plot(osm, slope*np.array(osm)+intercept, 'r--', lw=2)
ax.set_xlabel("Theoretical Quantiles"); ax.set_ylabel("Sample Quantiles")
ax.set_title("Q-Q Plot of Residuals", fontweight='bold')
plt.tight_layout()
plt.savefig("qq_plot.png", dpi=150, bbox_inches='tight')
plt.show()

stat, p_sw = stats.shapiro(residuals)
print(f"Shapiro-Wilk test: W={stat:.4f}, p={p_sw:.4f}")
if p_sw > 0.05:
    print("  → Residuals appear normally distributed (p > 0.05)")
else:
    print("  → Residuals deviate from normality (p ≤ 0.05)")

In [ ]:
# ── Qualitative prediction examples ───────────────────────────────────────────
n_show = 6
idx = np.random.RandomState(SEED).choice(len(X_test), n_show, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(14, 9))

for ax, i in zip(axes.flat, idx):
    img   = X_test[i]                         # already in [0,1]
    act   = y_true_mpa[i]
    pred_v = y_pred_mpa[i]
    err   = abs(act - pred_v)
    pct   = err / act * 100

    ax.imshow(img)
    ax.set_title(
        f"Actual: {act:.1f} MPa\n"
        f"Pred  : {pred_v:.1f} MPa\n"
        f"Error : {err:.1f} MPa ({pct:.1f}%)",
        fontsize=9
    )
    ax.axis('off')

plt.suptitle("Prediction Examples on Test Samples",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("prediction_examples.png", dpi=150, bbox_inches='tight')
plt.show()

## 14. Metrics Summary Table (IEEE-Ready)

In [ ]:
metrics_df = pd.DataFrame({
    "Metric" : ["MAE (MPa)", "RMSE (MPa)", "NRMSE (%)", "MAPE (%)", "R²"],
    "Value"  : [f"{mae:.4f}", f"{rmse:.4f}", f"{nrmse:.2f}",
                f"{mape:.2f}", f"{r2:.4f}"],
    "Description": [
        "Mean Absolute Error",
        "Root Mean Squared Error",
        "Normalised RMSE (as % of range)",
        "Mean Absolute Percentage Error",
        "Coefficient of Determination"
    ]
})

print("\n" + "=" * 62)
print("   SUMMARY OF TEST-SET PERFORMANCE (MobileNetV2 Regression)")
print("=" * 62)
print(metrics_df.to_string(index=False))
print("=" * 62)

# Save as CSV for LaTeX table generation
metrics_df.to_csv("metrics_summary.csv", index=False)
print("\nMetrics saved to metrics_summary.csv")

## 15. Save Model & Artefacts

In [ ]:
import joblib

# Save in modern Keras format (preferred over legacy .h5)
model.save("cfrp_cai_mobilenetv2.keras")

# Save scaler for inference
joblib.dump(scaler, "cai_strength_scaler.pkl")

print("Model saved  : cfrp_cai_mobilenetv2.keras")
print("Scaler saved : cai_strength_scaler.pkl")

## 16. Inference Helper (Deployment-Ready)

In [ ]:
def predict_cai_strength(image_path: str,
                          model: tf.keras.Model,
                          scaler: MinMaxScaler,
                          img_size: int = 224) -> float:
    """
    Predict CAI strength (MPa) from a single CFRP specimen image.

    Parameters
    ----------
    image_path : str
        Full path to the specimen image file.
    model : tf.keras.Model
        Trained regression model.
    scaler : MinMaxScaler
        Fitted label scaler.
    img_size : int
        Target spatial resolution (default: 224).

    Returns
    -------
    float
        Predicted CAI strength in MPa.
    """
    img = cv2.imread(image_path)
    if img is None:
        raise FileNotFoundError(f"Image not found: {image_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_AREA)
    img = img.astype(np.float32) / 255.0
    img = np.expand_dims(img, axis=0)             # (1, H, W, 3)

    pred_scaled = model.predict(img, verbose=0)   # (1, 1)
    pred_scaled = np.clip(pred_scaled, 0.0, 1.0)
    pred_mpa    = scaler.inverse_transform(pred_scaled)[0, 0]
    return float(pred_mpa)


# ── Demo on a random test image ───────────────────────────────────────────────
sample_idx  = np.random.randint(len(test_df))
sample_path = test_df.iloc[sample_idx]["img_path"]
sample_true = y_true_mpa[sample_idx]

predicted = predict_cai_strength(sample_path, model, scaler)
print(f"Sample image : {os.path.basename(sample_path)}")
print(f"Actual       : {sample_true:.2f} MPa")
print(f"Predicted    : {predicted:.2f} MPa")
print(f"Abs Error    : {abs(sample_true - predicted):.2f} MPa")